# Session 9 — Baseline model + formalized preprocessing (sklearn Pipeline)

Per `docs/Credit_Risk_Pipeline_Plan_v3.md` (buổi 9).

**Goal**: Logistic Regression baselines for two feature sets — `portfolio` (keeps the
underwriting columns) and `at_application` (drops them) — compared on ROC-AUC and PR-AUC.
The gap between them is the measurable cost of refusing to leak.

Preprocessing lives in `model/preprocessing.py::build_pipeline` (`StandardScaler` +
`OneHotEncoder` inside one `ColumnTransformer`), so it is fit exactly once, on the
training split only, and the same object is reused by sessions 10-12.

In [1]:
import sys

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

sys.path.insert(0, '../etl')
sys.path.insert(0, '..')
from config import get_engine
from model.features import (
    TARGET_COL,
    get_feature_columns,
    split_numeric_categorical,
)
from model.preprocessing import build_pipeline

engine = get_engine()

## Load data

Reads from the `ml_features` view (`sql/views.sql`) — already filtered to `data_source = 'historical'` and structurally excludes the dashboard-only window columns.

In [2]:
df = pd.read_sql("SELECT * FROM ml_features", engine)
print(df.shape)
df.head()

(32581, 25)


,loan_id,client_id,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,debt_to_income_ratio,loan_status,age,...,credit_utilization_ratio,past_delinquencies,country,loan_term_months,other_debt,gender,marital_status,education_level,employment_type,open_accounts
0,4,CUST_00001,PERSONAL,D,35000.0,16.02,0.59,0.7356,1,22,...,0.4956,0,Canada,36,8402.45,Male,Married,High School,Self-employed,14
1,5,CUST_00002,EDUCATION,B,1000.0,11.14,0.10,0.2716,0,21,...,0.5854,3,Canada,36,1607.80,Female,Divorced,Master,Full-time,10
2,6,CUST_00003,MEDICAL,C,5500.0,12.87,0.57,0.8605,1,25,...,0.7507,0,UK,36,2760.51,Female,Married,Master,Full-time,14
3,7,CUST_00004,MEDICAL,C,35000.0,15.23,0.53,0.6436,1,23,...,0.3793,0,Canada,12,7155.29,Male,Married,Bachelor,Part-time,15
4,8,CUST_00005,MEDICAL,C,35000.0,14.27,0.55,0.9306,1,24,...,0.2281,0,USA,36,15626.15,Female,Single,Bachelor,Part-time,4


## Feature sets

Recap of the buổi 8 decision (see `notebooks/01_eda.ipynb`, `CLAUDE.md`,
`docs/MEMORY.md`): `loan_grade` and `loan_int_rate` are near-deterministic with the
target and are outputs of underwriting, not inputs available at application time — they
stay in `LEAKAGE_COLS` (`model/features.py`) and are dropped from the
`at_application` feature set only.

In [3]:
feature_cols = {
    "portfolio": get_feature_columns(df.columns, "portfolio"),
    "at_application": get_feature_columns(df.columns, "at_application"),
}
for name, cols in feature_cols.items():
    print(f"{name} ({len(cols)} cols): {cols}")

portfolio (20 cols): ['loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'debt_to_income_ratio', 'age', 'income', 'home_ownership', 'emp_length', 'default_on_file', 'cred_hist_length', 'credit_utilization_ratio', 'past_delinquencies', 'country', 'loan_term_months', 'other_debt', 'education_level', 'employment_type', 'open_accounts']
at_application (18 cols): ['loan_intent', 'loan_amnt', 'loan_percent_income', 'debt_to_income_ratio', 'age', 'income', 'home_ownership', 'emp_length', 'default_on_file', 'cred_hist_length', 'credit_utilization_ratio', 'past_delinquencies', 'country', 'loan_term_months', 'other_debt', 'education_level', 'employment_type', 'open_accounts']


## Train/test split

One stratified split on `loan_status`, reused for both feature sets so the comparison
isn't confounded by different train/test rows.

In [4]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[TARGET_COL]
)
print(f"train: {train_df.shape}, test: {test_df.shape}")

train: (26064, 25), test: (6517, 25)


## Preprocessing rationale

The two decisions behind `model/preprocessing.py` — why `StandardScaler` fits
only after the split, and why `OneHotEncoder` rather than `LabelEncoder` — are
documented in that module's docstring, next to the code they explain.


## Train + evaluate both feature sets

In [5]:
results = {}

for name, cols in feature_cols.items():
    numeric_cols, categorical_cols = split_numeric_categorical(cols)
    pipeline = build_pipeline(numeric_cols, categorical_cols)
    pipeline.fit(train_df[cols], train_df[TARGET_COL])

    proba = pipeline.predict_proba(test_df[cols])[:, 1]
    results[name] = {
        "roc_auc": roc_auc_score(test_df[TARGET_COL], proba),
        "pr_auc": average_precision_score(test_df[TARGET_COL], proba),
    }

pd.DataFrame(results).T

,roc_auc,pr_auc
portfolio,0.871252,0.720649
at_application,0.807215,0.623982


## Comparison

| Feature set | ROC-AUC | PR-AUC | Features |
|---|---|---|---|
| `portfolio` | **0.8713** | **0.7206** | 20 |
| `at_application` | **0.8072** | **0.6240** | 18 |
| *gap* | *0.0640* | *0.0966* | *2 (`loan_grade`, `loan_int_rate`)* |

**Reading these numbers.** The base rate is ~21.8% defaults, which is the PR-AUC a random
classifier would score. Both models land far above it (0.72 and 0.62), so both have real
signal — neither is a dressed-up coin flip.

The gap is the leakage premium: two columns buy 0.064 ROC-AUC and 0.097 PR-AUC. Big enough
to be tempting, which is exactly why it needs refusing — `loan_grade` and `loan_int_rate`
are produced *by* underwriting, so a model using them can't run at application time, when
the decision actually has to be made. The at-application model is the honest one and is
what the session 13 Streamlit form will call.

Worth noting PR-AUC drops more than ROC-AUC (0.097 vs 0.064). PR-AUC is the stricter
metric on imbalanced data because it ignores true negatives, and predicting the 21.8%
minority correctly is the whole point here — this is the reason session 11 picks PR-AUC
as the deciding metric rather than ROC-AUC.

**Ceiling caveat**: session 8's scan found ~9 of these features are statistically noise
(`past_delinquencies` AUC 0.5004, `open_accounts` 0.4980, `credit_utilization_ratio`
0.5051, plus the demographic columns). In a real bureau file those would be among the
strongest predictors; here they were clearly generated independently of the target. So
0.807 is close to what this dataset can support at-application — session 10's XGBoost
should be expected to add a few points, not transform the result.